# Notebook 02 — EDA & Segmentation
**Dataset:** IBM Telco Customer Churn — `df_clean.parquet`  
**Tujuan:** Analisis 5 area faktor churn + segmentasi 3 kelompok risiko.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
from pathlib import Path

OUT     = Path('../output')
FIGURES = OUT / 'figures'
FIGURES.mkdir(exist_ok=True)

BLUE  = '#2563EB'
RED   = '#DC2626'
GRAY  = '#6B7280'
LIGHT = '#93C5FD'
sns.set_theme(style='whitegrid', font_scale=1.1)

df = pd.read_parquet(OUT / 'df_clean.parquet')
print(f'Loaded: {len(df):,} baris | Churn rate: {df["Churn"].mean():.1%}')

---
## A. Churn Overview

In [ ]:
churn_counts = df['Churn'].value_counts()
labels = ['Tidak Churn', 'Churn']
colors = [LIGHT, RED]

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# Donut
axes[0].pie(churn_counts.values, labels=labels, autopct='%1.1f%%',
            colors=colors, wedgeprops={'width': 0.55},
            startangle=90, textprops={'fontsize': 11})
axes[0].set_title('Proporsi Churn vs Retensi', fontweight='bold')

# Count bar
axes[1].bar(labels, churn_counts.values, color=colors, edgecolor='white', linewidth=0.5)
axes[1].set_ylabel('Jumlah Pelanggan')
axes[1].set_title('Distribusi Churn', fontweight='bold')
for i, v in enumerate(churn_counts.values):
    axes[1].text(i, v + 30, f'{v:,}', ha='center', fontweight='bold')

plt.suptitle(f'Churn Rate Keseluruhan: {df["Churn"].mean():.1%}',
             fontsize=13, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig(FIGURES / 'A_churn_overview.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: A_churn_overview.png')

---
## B. Faktor Kontrak & Tenure

In [ ]:
churn_contract = df.groupby('Contract')['Churn'].mean().sort_values(ascending=False)

seg_order = ['New (<12 bln)', 'Mid (12-24 bln)', 'Loyal (>24 bln)']
churn_tenure = df.groupby('tenure_segment')['Churn'].mean().reindex(seg_order)

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Churn by Contract
bar_colors = [RED if v == churn_contract.max() else LIGHT for v in churn_contract.values]
axes[0].bar(churn_contract.index, churn_contract.values * 100, color=bar_colors)
axes[0].set_ylabel('Churn Rate (%)')
axes[0].set_title('Churn Rate per Jenis Kontrak', fontweight='bold')
axes[0].yaxis.set_major_formatter(mticker.PercentFormatter())
for i, v in enumerate(churn_contract.values):
    axes[0].text(i, v * 100 + 0.5, f'{v:.1%}', ha='center', fontweight='bold')

# Churn by Tenure Segment
bar_colors2 = [RED if v == churn_tenure.max() else LIGHT for v in churn_tenure.values]
axes[1].bar(churn_tenure.index, churn_tenure.values * 100, color=bar_colors2)
axes[1].set_ylabel('Churn Rate (%)')
axes[1].set_title('Churn Rate per Segmen Tenure', fontweight='bold')
axes[1].yaxis.set_major_formatter(mticker.PercentFormatter())
for i, v in enumerate(churn_tenure.values):
    axes[1].text(i, v * 100 + 0.5, f'{v:.1%}', ha='center', fontweight='bold')

# KDE MonthlyCharges
df[df['Churn'] == 0]['MonthlyCharges'].plot.kde(ax=axes[2], color=BLUE, linewidth=2, label='Retensi')
df[df['Churn'] == 1]['MonthlyCharges'].plot.kde(ax=axes[2], color=RED,  linewidth=2, label='Churn')
axes[2].set_xlabel('Monthly Charges (USD)')
axes[2].set_title('Distribusi Monthly Charges\nChurn vs Retensi', fontweight='bold')
axes[2].legend()
axes[2].set_xlim(0, 120)

plt.tight_layout()
plt.savefig(FIGURES / 'B_contract_tenure.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: B_contract_tenure.png')

---
## C. Faktor Layanan

In [ ]:
# Churn rate by protective services (OnlineSecurity, TechSupport)
protect_cols = ['OnlineSecurity', 'TechSupport', 'OnlineBackup', 'DeviceProtection']
rows = []
for col in protect_cols:
    for val in ['Yes', 'No']:
        rate = df[df[col] == val]['Churn'].mean()
        rows.append({'service': col, 'has_service': val, 'churn_rate': rate})
service_df = pd.DataFrame(rows)

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Grouped bar — protective services
x = np.arange(len(protect_cols))
w = 0.35
yes_rates = service_df[service_df['has_service'] == 'Yes']['churn_rate'].values
no_rates  = service_df[service_df['has_service'] == 'No']['churn_rate'].values
axes[0].bar(x - w/2, yes_rates * 100, w, label='Punya Layanan', color=BLUE)
axes[0].bar(x + w/2, no_rates  * 100, w, label='Tidak Punya',   color=RED)
axes[0].set_xticks(x)
axes[0].set_xticklabels(protect_cols, rotation=15, ha='right')
axes[0].set_ylabel('Churn Rate (%)')
axes[0].yaxis.set_major_formatter(mticker.PercentFormatter())
axes[0].set_title('Churn Rate: Punya vs Tidak Punya Layanan Proteksi', fontweight='bold')
axes[0].legend()

# Churn by num_services
num_svc_churn = df.groupby('num_services')['Churn'].mean()
bar_colors3 = [RED if v > 0.3 else LIGHT for v in num_svc_churn.values]
axes[1].bar(num_svc_churn.index.astype(str), num_svc_churn.values * 100, color=bar_colors3)
axes[1].set_xlabel('Jumlah Layanan Aktif')
axes[1].set_ylabel('Churn Rate (%)')
axes[1].yaxis.set_major_formatter(mticker.PercentFormatter())
axes[1].set_title('Churn Rate vs Jumlah Layanan Aktif', fontweight='bold')
for i, v in enumerate(num_svc_churn.values):
    axes[1].text(i, v * 100 + 0.3, f'{v:.0%}', ha='center', fontsize=8)

plt.tight_layout()
plt.savefig(FIGURES / 'C_service_analysis.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: C_service_analysis.png')

---
## D. Faktor Finansial & Pembayaran

In [ ]:
churn_payment = df.groupby('PaymentMethod')['Churn'].mean().sort_values(ascending=False)

fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# Churn by payment method
bar_colors4 = [RED if v == churn_payment.max() else LIGHT for v in churn_payment.values]
axes[0].barh(churn_payment.index[::-1], churn_payment.values[::-1] * 100,
             color=bar_colors4[::-1])
axes[0].set_xlabel('Churn Rate (%)')
axes[0].xaxis.set_major_formatter(mticker.PercentFormatter())
axes[0].set_title('Churn Rate per Metode Pembayaran', fontweight='bold')
for i, v in enumerate(churn_payment.values[::-1]):
    axes[0].text(v * 100 + 0.3, i, f'{v:.1%}', va='center')

# Scatter MonthlyCharges vs tenure
churn_mask = df['Churn'] == 1
axes[1].scatter(df[~churn_mask]['tenure'], df[~churn_mask]['MonthlyCharges'],
                alpha=0.15, color=BLUE, s=10, label='Retensi')
axes[1].scatter(df[churn_mask]['tenure'],  df[churn_mask]['MonthlyCharges'],
                alpha=0.35, color=RED,  s=10, label='Churn')
axes[1].set_xlabel('Tenure (bulan)')
axes[1].set_ylabel('Monthly Charges (USD)')
axes[1].set_title('Monthly Charges vs Tenure\n(merah = Churn)', fontweight='bold')
axes[1].legend(markerscale=3)

plt.tight_layout()
plt.savefig(FIGURES / 'D_financial_payment.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: D_financial_payment.png')

---
## E. Risk Segment Summary

In [ ]:
seg_order = ['High Risk', 'Mid Risk', 'Low Risk']
seg_summary = (
    df.groupby('risk_segment')
    .agg(
        jumlah_pelanggan=('Churn', 'count'),
        churn_rate=('Churn', 'mean'),
        avg_monthly_charges=('MonthlyCharges', 'mean'),
        avg_tenure=('tenure', 'mean')
    )
    .reindex(seg_order)
    .reset_index()
)

print('\n=== Risk Segment Summary ===')
print(seg_summary.to_string(index=False))

# Bar chart churn rate per segmen
seg_colors = [RED, '#F59E0B', BLUE]
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].bar(seg_summary['risk_segment'], seg_summary['churn_rate'] * 100,
            color=seg_colors)
axes[0].set_ylabel('Churn Rate (%)')
axes[0].yaxis.set_major_formatter(mticker.PercentFormatter())
axes[0].set_title('Churn Rate per Risk Segment', fontweight='bold')
for i, v in enumerate(seg_summary['churn_rate']):
    axes[0].text(i, v * 100 + 0.5, f'{v:.1%}', ha='center', fontweight='bold')

axes[1].bar(seg_summary['risk_segment'], seg_summary['jumlah_pelanggan'],
            color=seg_colors)
axes[1].set_ylabel('Jumlah Pelanggan')
axes[1].set_title('Jumlah Pelanggan per Risk Segment', fontweight='bold')
for i, v in enumerate(seg_summary['jumlah_pelanggan']):
    axes[1].text(i, v + 20, f'{v:,}', ha='center', fontweight='bold')

plt.tight_layout()
plt.savefig(FIGURES / 'E_risk_segments.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: E_risk_segments.png')